<a href="https://colab.research.google.com/github/Saliyah-53/saliyah-stanceeval2026/blob/main/SaliAI_Track1_Synthetic_Fanar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SaliAI — Track 1: Synthetic Data Generation with Fanar + Model Training

Instead of distilling on only 352 tweets (which failed), we let **Fanar generate
thousands of labelled tweets** across **diverse topics**, and train MARBERT on
(gold training data + Fanar's examples). Wider variety may help generalization.

- Fanar generates tweets + their stance for dozens of topics
- MARBERT trains on all of it (20 epochs, best checkpoint saved)
- May beat Fanar by seeing broader variety

Saves to a new Drive folder. Requires a T4 GPU. Run all.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece arabert


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.0/185.0 kB 11.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 108.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 13.8 MB/s eta 0:00:00


In [ ]:
# ============================================================
# GPU CHECK
# ============================================================
import torch

print('GPU جاهز:', torch.cuda.get_device_name(0))


GPU جاهز: Tesla T4


In [ ]:
# ============================================================
# 1) Install Required Packages
# ============================================================
!pip install -q transformers scikit-learn pandas tqdm sentencepiece
!pip install -q arabert


In [ ]:
# ============================================================
# 2) Imports
# ============================================================
import os
import re
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import f1_score, accuracy_score, classification_report
from tqdm.auto import tqdm

from arabert.preprocess import ArabertPreprocessor
arabert_prep = ArabertPreprocessor(model_name="UBC-NLP/MARBERTv2", keep_emojis=True)


In [ ]:
# ============================================================
# 3) Configuration
# ============================================================
BASE_DIR = "."
DATA_DIR = f"{BASE_DIR}/data"

TRAIN_PATH = f"{DATA_DIR}/train.csv"
DEV_PATH = f"{DATA_DIR}/dev.csv"
TEST_SEEN_PATH = f"{DATA_DIR}/test_seen.csv"
TEST_UNSEEN_PATH = f"{DATA_DIR}/test_unseen.csv"

OUTPUT_DIR = f"{BASE_DIR}/saliai_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Columns
TEXT_COL = "text"
TARGET_COL = "target"
LABEL_COL = "stance"
ID_COL = "id"
SENT_COL = "sentiment"
SARC_COL = "sarcasm"
CONF_COL = "stance:confidence"
FAVOR_REASON_COL = "favor_reason"
AGAINST_REASON_COL = "against_reason"

# Model
MODEL_DISPLAY_NAME = "SaliAI-MARBERTv2-MTL"
MODEL_HF_NAME = "UBC-NLP/MARBERTv2"

MAX_LEN = 128
BATCH_SIZE = 32
EPOCHS = 10
LR = 2e-5
WARMUP_RATIO = 0.1
SEED = 42

# Labels
LABEL2ID = {"Against": 0, "Favor": 1, "None": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
SENT2ID = {"Negative": 0, "Neutral": 1, "Positive": 2}
SARC2ID = {"No": 0, "Yes": 1}
IGNORE = -100

# Auxiliary loss weights (stance loss weight = 1.0)
AUX_W_SENT = 0.3
AUX_W_SARC = 0.2
AUX_W_EXPL = 0.2

# Confidence weighting: sample_weight = 0.5 + 0.5 * confidence
CONF_FLOOR = 0.5

required_files = [TRAIN_PATH, DEV_PATH]
for path in required_files:
    if not os.path.exists(path):
        raise FileNotFoundError(f"File not found: {path}. Put train.csv and dev.csv inside the data/ folder.")

print("All required dataset files found.")
print("Model:", MODEL_DISPLAY_NAME)


All required dataset files found.
Model: SaliAI-MARBERTv2-MTL


In [ ]:
# ============================================================
# 4) Reproducibility
# ============================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)


Using device: cuda


In [ ]:
# ============================================================
# Arabic Text Preprocessing -- official AraBERT/MARBERT preprocessor
# ============================================================
# يعالج المعرّفات (MENTION) والروابط (URL) والإيموجي والهمزات والأرقام والهاشتاقات
# بشكل مضبوط خصيصاً للنموذج، بدل التنظيف اليدوي.

def preprocess_text(text):
    text = str(text)
    try:
        text = arabert_prep.preprocess(text)
    except Exception:
        pass
    return text.strip()


In [ ]:
# ============================================================
# 6) Load Data + Build Auxiliary Labels
# ============================================================
def add_aux_labels(df):
    # Sentiment
    if SENT_COL in df.columns:
        df["sent_label"] = df[SENT_COL].map(lambda v: SENT2ID.get(str(v).strip(), IGNORE))
    else:
        df["sent_label"] = IGNORE

    # Sarcasm
    if SARC_COL in df.columns:
        df["sarc_label"] = df[SARC_COL].map(lambda v: SARC2ID.get(str(v).strip(), IGNORE))
    else:
        df["sarc_label"] = IGNORE

    # Stance explicitness (from reason columns: *_Explicit / *_Implicit)
    def expl_lab(row):
        for col in (FAVOR_REASON_COL, AGAINST_REASON_COL):
            v = str(row.get(col, "")).strip()
            if v.endswith("Explicit"):
                return 0
            if v.endswith("Implicit"):
                return 1
        return IGNORE

    if FAVOR_REASON_COL in df.columns or AGAINST_REASON_COL in df.columns:
        df["expl_label"] = df.apply(expl_lab, axis=1)
    else:
        df["expl_label"] = IGNORE

    # Annotator-confidence sample weight
    def conf_weight(v):
        try:
            c = float(v)
        except (TypeError, ValueError):
            c = 1.0
        return CONF_FLOOR + (1.0 - CONF_FLOOR) * c

    if CONF_COL in df.columns:
        df["sample_weight"] = df[CONF_COL].map(conf_weight)
    else:
        df["sample_weight"] = 1.0

    return df


def load_labeled_dataframe(path):
    df = pd.read_csv(path, keep_default_na=False)
    needed = [TEXT_COL, TARGET_COL, LABEL_COL]
    for c in needed:
        if c not in df.columns:
            raise ValueError(f"Missing column: {c}")
        df[c] = df[c].astype(str).str.strip()
    df = df[(df[TEXT_COL] != "") & (df[TARGET_COL] != "") & (df[LABEL_COL] != "")].copy()
    df[TEXT_COL] = df[TEXT_COL].apply(preprocess_text)
    unknown = sorted(set(df[LABEL_COL]) - set(LABEL2ID.keys()))
    if unknown:
        raise ValueError(f"Unknown stance labels: {unknown}")
    df["label"] = df[LABEL_COL].map(LABEL2ID).astype(int)
    df = add_aux_labels(df)
    return df


def load_unlabeled_test_dataframe(path):
    df = pd.read_csv(path, keep_default_na=False)
    needed = [ID_COL, TEXT_COL, TARGET_COL]
    for c in needed:
        if c not in df.columns:
            raise ValueError(f"Missing column: {c}")
        df[c] = df[c].astype(str).str.strip()
    df = df[(df[ID_COL] != "") & (df[TEXT_COL] != "") & (df[TARGET_COL] != "")].copy()
    df[TEXT_COL] = df[TEXT_COL].apply(preprocess_text)
    return df


train_df = load_labeled_dataframe(TRAIN_PATH)
dev_df = load_labeled_dataframe(DEV_PATH)

print("Train shape:", train_df.shape)
print("Dev shape:", dev_df.shape)
print("\nTrain stance counts:", train_df[LABEL_COL].value_counts().to_dict())
print("Aux coverage -> sentiment:", (train_df['sent_label'] != IGNORE).sum(),
      "| sarcasm:", (train_df['sarc_label'] != IGNORE).sum(),
      "| explicitness:", (train_df['expl_label'] != IGNORE).sum())
print("Sample weight range: [{:.3f}, {:.3f}]".format(train_df['sample_weight'].min(), train_df['sample_weight'].max()))


Train shape: (3502, 19)
Dev shape: (619, 19)

Train stance counts: {'Favor': 2148, 'Against': 1021, 'None': 333}
Aux coverage -> sentiment: 3502 | sarcasm: 3502 | explicitness: 3110
Sample weight range: [0.677, 1.000]


In [ ]:
# ============================================================
# 7) Dataset
# ============================================================
class StanceDataset(Dataset):
    def __init__(self, df, tokenizer, is_test=False):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(
            row[TARGET_COL],
            row[TEXT_COL],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if not self.is_test:
            item["labels"] = torch.tensor(int(row["label"]), dtype=torch.long)
            item["sent_label"] = torch.tensor(int(row["sent_label"]), dtype=torch.long)
            item["sarc_label"] = torch.tensor(int(row["sarc_label"]), dtype=torch.long)
            item["expl_label"] = torch.tensor(int(row["expl_label"]), dtype=torch.long)
            item["sample_weight"] = torch.tensor(float(row["sample_weight"]), dtype=torch.float)
        return item


In [ ]:
# ============================================================
# 8) Multi-Task Model + Loss
# ============================================================
class SaliAIModel(nn.Module):
    def __init__(self, hf_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(hf_name)
        h = self.encoder.config.hidden_size
        self.dropout = nn.Dropout(0.2)
        self.stance_head = nn.Linear(h, 3)
        self.sent_head = nn.Linear(h, 3)
        self.sarc_head = nn.Linear(h, 2)
        self.expl_head = nn.Linear(h, 2)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        kwargs = {"input_ids": input_ids, "attention_mask": attention_mask}
        if token_type_ids is not None:
            kwargs["token_type_ids"] = token_type_ids
        out = self.encoder(**kwargs)
        pooled = self.dropout(out.last_hidden_state[:, 0])
        return {
            "stance": self.stance_head(pooled),
            "sent": self.sent_head(pooled),
            "sarc": self.sarc_head(pooled),
            "expl": self.expl_head(pooled),
        }


# Class weights from training distribution (inverse frequency)
counts = train_df["label"].value_counts().reindex([0, 1, 2]).fillna(0).values.astype(float)
class_weights = torch.tensor(len(train_df) / (3.0 * np.maximum(counts, 1.0)), dtype=torch.float).to(DEVICE)
print("Class weights (Against, Favor, None):", [round(w, 3) for w in class_weights.tolist()])


def safe_aux_ce(logits, labels):
    if (labels != IGNORE).any():
        return F.cross_entropy(logits, labels, ignore_index=IGNORE)
    return torch.tensor(0.0, device=logits.device)


def compute_loss(logits, batch):
    per_sample = F.cross_entropy(logits["stance"], batch["labels"], weight=class_weights, reduction="none")
    loss = (per_sample * batch["sample_weight"]).mean()
    loss = loss + AUX_W_SENT * safe_aux_ce(logits["sent"], batch["sent_label"])
    loss = loss + AUX_W_SARC * safe_aux_ce(logits["sarc"], batch["sarc_label"])
    loss = loss + AUX_W_EXPL * safe_aux_ce(logits["expl"], batch["expl_label"])
    return loss


Class weights (Against, Favor, None): [1.143, 0.543, 3.506]


In [ ]:
# ============================================================
# 9) Tokenizer, Loaders, Optimizer
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_HF_NAME)

train_ds = StanceDataset(train_df, tokenizer)
dev_ds = StanceDataset(dev_df, tokenizer)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False)

model = SaliAIModel(MODEL_HF_NAME).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LR)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(WARMUP_RATIO * total_steps),
    num_training_steps=total_steps,
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/757 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/439 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/1.10M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  654MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# ============================================================
# 10) Metrics + Prediction Helpers
# ============================================================
ENC_KEYS = ("input_ids", "attention_mask", "token_type_ids")

def enc_batch(batch):
    return {k: batch[k].to(DEVICE) for k in ENC_KEYS if k in batch}


def compute_metrics(y_true, y_pred):
    f_against = f1_score(y_true, y_pred, labels=[0], average="macro", zero_division=0)
    f_favor = f1_score(y_true, y_pred, labels=[1], average="macro", zero_division=0)
    f_none = f1_score(y_true, y_pred, labels=[2], average="macro", zero_division=0)
    return {
        "F_favor": f_favor,
        "F_against": f_against,
        "F_none": f_none,
        "Favg2": (f_favor + f_against) / 2.0,
        "Favg3": (f_favor + f_against + f_none) / 3.0,
        "Acc": accuracy_score(y_true, y_pred),
    }


@torch.no_grad()
def predict_labeled_loader(model, loader):
    model.eval()
    preds, labels = [], []
    for batch in loader:
        labels.extend(batch["labels"].numpy().tolist())
        logits = model(**enc_batch(batch))["stance"]
        preds.extend(torch.argmax(logits, dim=1).cpu().numpy().tolist())
    return labels, preds


@torch.no_grad()
def predict_unlabeled_loader(model, loader):
    model.eval()
    preds = []
    for batch in tqdm(loader, desc="Predicting"):
        logits = model(**enc_batch(batch))["stance"]
        preds.extend(torch.argmax(logits, dim=1).cpu().numpy().tolist())
    return preds


def build_per_target_results(eval_df, pred_ids, model_name):
    df = eval_df.copy()
    df["pred"] = pred_ids
    row = {"Model": model_name}
    for target in sorted(df[TARGET_COL].unique()):
        sub = df[df[TARGET_COL] == target]
        m = compute_metrics(sub["label"].tolist(), sub["pred"].tolist())
        key = target.replace(" ", "_")
        row[f"{key}_Favg2"] = round(m["Favg2"] * 100, 2)
        row[f"{key}_Favg3"] = round(m["Favg3"] * 100, 2)
    m = compute_metrics(df["label"].tolist(), df["pred"].tolist())
    row["F_favor"] = round(m["F_favor"] * 100, 2)
    row["F_against"] = round(m["F_against"] * 100, 2)
    row["F_none"] = round(m["F_none"] * 100, 2)
    row["Overall_Favg2"] = round(m["Favg2"] * 100, 2)
    row["Overall_Favg3"] = round(m["Favg3"] * 100, 2)
    row["Acc"] = round(m["Acc"] * 100, 2)
    return pd.DataFrame([row])


In [ ]:
import pandas as pd, numpy as np, os, torch, zipfile, torch.nn.functional as F
from copy import deepcopy
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

EPOCHS = 20
train_plus_dev = pd.concat([train_df, dev_df], ignore_index=True)
print("train:", len(train_df), "| dev:", len(dev_df), "| المجموع:", len(train_plus_dev))

rng=np.random.RandomState(42)
idx=rng.permutation(len(train_plus_dev)); cut=int(0.9*len(idx))
tr_idx, va_idx = idx[:cut], idx[cut:]
tr_df = train_plus_dev.iloc[tr_idx].reset_index(drop=True)
va_df = train_plus_dev.iloc[va_idx].reset_index(drop=True)
print("تدريب:", len(tr_df), "| تحقق داخلي:", len(va_df))


def load_test(path):
    d=pd.read_csv(path, keep_default_na=False)
    if "tweet_text" in d.columns and TEXT_COL not in d.columns: d=d.rename(columns={"tweet_text":TEXT_COL})
    for c in [TEXT_COL, TARGET_COL]: d[c]=d[c].astype(str).str.strip()
    d[TEXT_COL]=d[TEXT_COL].apply(preprocess_text)
    return d
test_df = load_test(f"{DATA_DIR}/test_seen.csv")
print("اختبار:", len(test_df))

train: 3502 | dev: 619 | المجموع: 4121
تدريب: 3708 | تحقق داخلي: 413
اختبار: 352


In [ ]:
def train_student(train_frame, tag):
    m = SaliAIModel(MODEL_HF_NAME).to(DEVICE)
    trl=DataLoader(StanceDataset(train_frame, tokenizer), batch_size=BATCH_SIZE, shuffle=True)
    val=DataLoader(StanceDataset(va_df, tokenizer), batch_size=BATCH_SIZE, shuffle=False)
    opt=AdamW(m.parameters(), lr=LR); tot=len(trl)*EPOCHS
    sch=get_linear_schedule_with_warmup(opt, int(WARMUP_RATIO*tot), tot)
    best=-1; bs=None
    for ep in range(EPOCHS):
        m.train()
        for b in tqdm(trl, desc=f"{tag} ep{ep+1}/{EPOCHS}"):
            opt.zero_grad()
            loss=compute_loss(m(**enc_batch(b)),
                {k:b[k].to(DEVICE) for k in ("labels","sent_label","sarc_label","expl_label","sample_weight")})
            loss.backward(); torch.nn.utils.clip_grad_norm_(m.parameters(),1.0); opt.step(); sch.step()
        dl,dp=predict_labeled_loader(m,val); mm=compute_metrics(dl,dp)
        if mm["Favg2"]>best: best=mm["Favg2"]; bs=deepcopy(m.state_dict())
    m.load_state_dict(bs)
    print(f"{tag}: أفضل تحقق داخلي Favg2 = {best*100:.2f}")
    return m

def predict_test(m):
    tl=DataLoader(StanceDataset(test_df, tokenizer, is_test=True), batch_size=BATCH_SIZE, shuffle=False)
    m.eval(); ids=[]
    with torch.no_grad():
        for b in tqdm(tl, desc="predict test"):
            ids += m(**enc_batch(b))["stance"].argmax(1).cpu().tolist()
    return [ID2LABEL[i] for i in ids]

def save_sub(preds, name):
    txt=f"{OUTPUT_DIR}/{name}.txt"; open(txt,"w",encoding="utf-8").write("\n".join(preds)+"\n")
    with zipfile.ZipFile(f"{OUTPUT_DIR}/{name}.zip","w",zipfile.ZIP_DEFLATED) as z: z.write(txt, arcname="submission_seen.txt")
    print(f"{name}: {pd.Series(preds).value_counts().to_dict()}")

## Synthetic data generation with Fanar across diverse topics

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import gc, re, random
bnb=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                       bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
ftok=AutoTokenizer.from_pretrained("QCRI/Fanar-1-9B-Instruct", trust_remote_code=True)
if ftok.pad_token is None: ftok.pad_token=ftok.eos_token
fmdl=AutoModelForCausalLM.from_pretrained("QCRI/Fanar-1-9B-Instruct", quantization_config=bnb, device_map="auto", trust_remote_code=True)
fmdl.eval()

def fgen(prompt, max_new=220, temp=0.9):
    m=[{"role":"user","content":prompt}]
    p=ftok.apply_chat_template(m, add_generation_prompt=True, tokenize=False)
    e=ftok(p, return_tensors="pt", return_token_type_ids=False).to(fmdl.device)
    with torch.no_grad():
        o=fmdl.generate(**e, max_new_tokens=max_new, do_sample=True, temperature=temp, top_p=0.95, pad_token_id=ftok.eos_token_id)
    return ftok.decode(o[0][e["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

TOPICS=["قيادة المرأة للسيارة","تمكين المرأة في العمل","التعليم عن بعد","الرياضة النسائية",
        "السياحة الداخلية","التحول للطاقة المتجددة","دوام العمل المرن","الذكاء الاصطناعي في التعليم",
        "المدن الذكية","ريادة الأعمال للشباب","الأمن الغذائي","العملات الرقمية"]
STANCES=[("Favor","مؤيد وداعم بوضوح"),("Against","معارض ورافض بوضوح"),("None","محايد أو مجرد خبر/سؤال دون موقف")]

def parse_tweets(txt):

    out=[]
    for line in txt.split("\n"):
        s=re.sub(r"^\s*[-•\d\.\)]+\s*","",line).strip()
        if 8 <= len(s) <= 280 and not s.startswith("الموقف") and "تغريدة" not in s[:6]:
            out.append(s)
    return out

SYN=[]
PER_CELL=6
for topic in tqdm(TOPICS, desc="topics"):
    for stance,desc in STANCES:
        prompt=(f"اكتب {PER_CELL} تغريدات عربية واقعية بلهجات مختلفة تعبّر عن موقف {desc} تجاه: «{topic}». "
                f"اجعلها متنوعة وقصيرة كتغريدات حقيقية. اكتب كل تغريدة في سطر منفصل دون ترقيم أو شرح.")
        try:
            tws=parse_tweets(fgen(prompt))
        except Exception as e:
            tws=[]
        for tw in tws[:PER_CELL]:
            SYN.append({"text":tw,"target":topic,"stance":stance})
print("عدد الأمثلة الصناعية المولّدة:", len(SYN))
import pandas as pd
syn_df=pd.DataFrame(SYN)
print(syn_df["stance"].value_counts().to_dict())
del fmdl; gc.collect(); torch.cuda.empty_cache()

config.json:   0%|          | 0.00/900 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/47.2k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 18.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/555 [00:00<?, ?B/s]

...

[transformers] The following generation flags are not valid and may be ignored: ['cache_implementation']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

topics:   0%|          | 0/12 [00:00<?, ?it/s]

عدد الأمثلة الصناعية المولّدة: 208
{'None': 70, 'Favor': 69, 'Against': 69}


In [ ]:

syn=syn_df.copy()
syn[TEXT_COL]=syn["text"].astype(str).apply(preprocess_text)
syn[TARGET_COL]=syn["target"].astype(str)
syn[LABEL_COL]=syn["stance"].astype(str)
syn["label"]=syn[LABEL_COL].map(LABEL2ID).astype(int)
syn["sent_label"]=IGNORE; syn["sarc_label"]=IGNORE; syn["expl_label"]=IGNORE
syn["sample_weight"]=0.6   # وزن أقل للبيانات الصناعية
cols=["label","sent_label","sarc_label","expl_label","sample_weight",TEXT_COL,TARGET_COL,LABEL_COL]

gold=train_df.copy()
for c in cols:
    if c not in gold.columns: gold[c]=IGNORE if "label" in c else (1.0 if c=="sample_weight" else "")
big_train=pd.concat([gold[cols], syn[cols]], ignore_index=True)
print("حجم التدريب الكلي:", len(big_train), "(ذهب:", len(gold), "+ صناعي:", len(syn), ")")


import numpy as np
rng=np.random.RandomState(42); idx=rng.permutation(len(big_train)); cut=int(0.9*len(idx))
BT=big_train.iloc[idx[:cut]].reset_index(drop=True)
VA=big_train.iloc[idx[cut:]].reset_index(drop=True)

حجم التدريب الكلي: 3710 (ذهب: 3502 + صناعي: 208 )


In [ ]:

from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from copy import deepcopy
EPOCHS=20
def train_on(frame, va, tag):
    m=SaliAIModel(MODEL_HF_NAME).to(DEVICE)
    trl=DataLoader(StanceDataset(frame, tokenizer), batch_size=BATCH_SIZE, shuffle=True)
    val=DataLoader(StanceDataset(va, tokenizer), batch_size=BATCH_SIZE, shuffle=False)
    opt=AdamW(m.parameters(), lr=LR); tot=len(trl)*EPOCHS
    sch=get_linear_schedule_with_warmup(opt, int(WARMUP_RATIO*tot), tot)
    best=-1; bs=None
    for ep in range(EPOCHS):
        m.train()
        for b in tqdm(trl, desc=f"{tag} ep{ep+1}/{EPOCHS}"):
            opt.zero_grad()
            loss=compute_loss(m(**enc_batch(b)),
                {k:b[k].to(DEVICE) for k in ("labels","sent_label","sarc_label","expl_label","sample_weight")})
            loss.backward(); torch.nn.utils.clip_grad_norm_(m.parameters(),1.0); opt.step(); sch.step()
        dl,dp=predict_labeled_loader(m,val); mm=compute_metrics(dl,dp)
        if mm["Favg2"]>best: best=mm["Favg2"]; bs=deepcopy(m.state_dict())
    m.load_state_dict(bs); print(f"{tag}: أفضل تحقق داخلي={best*100:.2f}")
    return m

def load_test(path):
    d=pd.read_csv(path, keep_default_na=False)
    if "tweet_text" in d.columns and TEXT_COL not in d.columns: d=d.rename(columns={"tweet_text":TEXT_COL})
    for c in [TEXT_COL,TARGET_COL]: d[c]=d[c].astype(str).str.strip()
    d[TEXT_COL]=d[TEXT_COL].apply(preprocess_text); return d
test_df=load_test(f"{DATA_DIR}/test_seen.csv")

modelS=train_on(BT, VA, "synthetic_aug")
tl=DataLoader(StanceDataset(test_df, tokenizer, is_test=True), batch_size=BATCH_SIZE, shuffle=False)
modelS.eval(); ids=[]
with torch.no_grad():
    for b in tqdm(tl, desc="predict"): ids += modelS(**enc_batch(b))["stance"].argmax(1).cpu().tolist()
preds=[ID2LABEL[i] for i in ids]

import zipfile
txt=f"{OUTPUT_DIR}/sub_synthetic_aug.txt"; open(txt,"w",encoding="utf-8").write("\n".join(preds)+"\n")
with zipfile.ZipFile(f"{OUTPUT_DIR}/sub_synthetic_aug.zip","w",zipfile.ZIP_DEFLATED) as z: z.write(txt, arcname="submission_seen.txt")
print("توزيع:", pd.Series(preds).value_counts().to_dict())
print(">>> ارفع sub_synthetic_aug على Track 1 وقارن مع Fanar (0.7152) <<<")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: UBC-NLP/MARBERTv2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


synthetic_aug ep1/20:   0%|          | 0/105 [00:00<?, ?it/s]

synthetic_aug ep2/20:   0%|          | 0/105 [00:00<?, ?it/s]

synthetic_aug ep3/20:   0%|          | 0/105 [00:00<?, ?it/s]

...

synthetic_aug ep19/20:   0%|          | 0/105 [00:00<?, ?it/s]

synthetic_aug ep20/20:   0%|          | 0/105 [00:00<?, ?it/s]

synthetic_aug: أفضل تحقق داخلي=85.55


predict:   0%|          | 0/11 [00:00<?, ?it/s]

توزيع: {'Against': 155, 'Favor': 131, 'None': 66}
>>> ارفع sub_synthetic_aug على Track 1 وقارن مع Fanar (0.7152) <<<
